1: Import libraries

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

2: Load the RFM dataset

In [ ]:
rfm = pd.read_csv("../data/processed/rfm.csv")

rfm.head()

3: Check for missing values


In [ ]:
rfm.info()
rfm.isnull().sum()

4: Scale the features


In [ ]:
scaler = StandardScaler()

rfm_scaled = scaler.fit_transform(
    rfm[["Recency", "Frequency", "Monetary"]]
)

5: Find the optimal number of clusters (Elbow Method)

In [ ]:
wcss = []

for i in range(2, 11):
    model = KMeans(n_clusters=i, random_state=42, n_init=10)
    model.fit(rfm_scaled)
    wcss.append(model.inertia_)

plt.figure(figsize=(8,5))
plt.plot(range(2,11), wcss, marker="o")
plt.xlabel("Number of Clusters")
plt.ylabel("WCSS")
plt.title("Elbow Method")
plt.show()

6: Calculate Silhouette Scores

In [ ]:
for k in range(2,11):
    model = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = model.fit_predict(rfm_scaled)

    score = silhouette_score(rfm_scaled, labels)
    print(f"k={k}: {score:.3f}")

7: Train the final model

In [ ]:
kmeans = KMeans(
    n_clusters=4,
    random_state=42,
    n_init=10
)

rfm["Cluster"] = kmeans.fit_predict(rfm_scaled)

 8: Explore the clusters


In [ ]:
rfm.groupby("Cluster")[["Recency","Frequency","Monetary"]].mean()

 9: Visualize the clusters

In [ ]:
plt.figure(figsize=(10,6))

sns.scatterplot(
    data=rfm,
    x="Frequency",
    y="Monetary",
    hue="Cluster",
    palette="Set2",
    s=80
)

plt.title("Customer Segments")
plt.show()

10: Save the segmented customers


In [ ]:
rfm.to_csv(
    "../data/processed/customer_segments.csv",
    index=False
)